# tp-10 オッズの挙動の年ごとの変化と、最終オッズ予測へのドリフト対策

**背景（ユーザーの指摘）**: 締切直前の大量購入（AI の参入など）でオッズが一気に変わるケースが増え、同じタイプの馬が以前は 20 倍だったのに最近は 10〜15 倍になっている印象がある。最終オッズ予測（`final_odds_*`、馬券推奨の「予測オッズ」）や馬券シミュレーションが、その影響を受けているのでは。

**確認できること / できないこと**
- 手元の 2020〜2025 年は**確定オッズだけ**で、T-60 / T-15 のスナップショットが無い（2026-10-17 以降に蓄積）。締切直前の動きそのものは、今の履歴からは測れない。
- ただし**確定オッズの年ごとの構造変化**（人気の集中、穴馬のオッズの水準、市場の歪み）は測れる。もし直前の資金流入が増えて穴馬が売れているなら、確定オッズの水準にも出るはず。

## 内容
1. 年別の市場構造（払戻率、本命の暗黙確率、オッズ帯ごとの確率の質量、人気順位別のオッズ中央値）。
2. 同じモデル確率の馬の確定オッズの年別変化（v2 の p_win 帯別）。
3. **最終オッズ予測のローリング評価**（学習 = 検証年より前、評価 = 検証年。2022〜2025 の 4 年）: 一様な重み / 新しい年を重く（半減期 1・2 年）/ 直近 2・3 年だけ。予測の偏り（残差の平均）が年を追ってずれるか、重み付けで直るかを調べる。特徴量は学習表の base+S+C1（本番の `final_odds` モデルの代理）。
4. 結論を `final_odds` の学習設定（`recency_halflife_years`）への提案として残す。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow
import lightgbm as lgb

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
OUT = WORK / "odds_drift"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP10_THREADS", detect_compute(reserve_cores=6).threads))
mlflow.set_experiment("keiba-tp-odds-drift")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
wp = pd.read_parquet(BET / "odds_win_place.parquet")
wp = wp[wp["win_odds"] > 0].copy()          # win_odds = -3.0 は取消・除外
wp["year"] = wp["race_id"].str[:4].astype(int)
pay = pd.read_parquet(BET / "payoff.parquet")
win_no = pay[(pay["bet_type"] == "tansho") & pay["combo"].notna()].groupby("race_id")["combo"].agg(["count", "first"])
win_no = win_no[win_no["count"] == 1]["first"].astype(int)
wp = wp[wp["race_id"].isin(win_no.index)].copy()
wp["won"] = wp["horse_number"].to_numpy() == win_no.reindex(wp["race_id"]).to_numpy()
inv = 1.0 / wp["win_odds"]
wp["q"] = inv / inv.groupby(wp["race_id"]).transform("sum")
wp["rank"] = wp.groupby("race_id")["win_odds"].rank(method="first")
rows = []
for y, g in wp.groupby("year"):
    band = lambda lo, hi: float(g.loc[(g["win_odds"] >= lo) & (g["win_odds"] < hi), "q"].sum() / g["q"].sum())
    rows.append({"year": y, "払戻率": float(1 / g.groupby("race_id")["win_odds"].apply(lambda s: (1 / s).sum()).mean()), "本命の暗黙確率": float(g.groupby("race_id")["q"].max().mean()),
                 "質量<5倍": band(1, 5), "質量5-10倍": band(5, 10), "質量10-20倍": band(10, 20), "質量20-50倍": band(20, 50), "質量50倍以上": band(50, 1e9),
                 "6-10番人気の中央値": float(g[(g["rank"] >= 6) & (g["rank"] <= 10)]["win_odds"].median()), "11番人気以下の中央値": float(g[g["rank"] >= 11]["win_odds"].median())})
STRUCT = pd.DataFrame(rows).set_index("year")
display(STRUCT.round(3)); STRUCT.to_csv(OUT / "market_structure_by_year.csv")
bins = [1, 3, 6, 10, 20, 50, 1e9]; lab = ["<3", "3-6", "6-10", "10-20", "20-50", "50+"]
wp["band"] = pd.cut(wp["win_odds"], bins, labels=lab, right=False)
cal = wp.groupby(["year", "band"], observed=True).agg(n=("won", "size"), 実勝率=("won", "mean"), 暗黙=("q", "mean")).reset_index()
cal["実/暗黙"] = cal["実勝率"] / cal["暗黙"]
display(cal.pivot(index="band", columns="year", values="実/暗黙").round(3)); cal.to_csv(OUT / "calibration_by_band_year.csv", index=False)

In [ ]:
# 同じモデル確率の馬の確定オッズ（v2 の p_win 帯別の中央値）
pr = pd.concat([pd.read_parquet(WORK / "oof/model_oof/ensemble_v2.parquet")[["race_id", "horse_id", "p_win"]], pd.read_parquet(WORK / "final_eval_v2/test2025_predictions.parquet")[["race_id", "horse_id", "p_win"]]])
m = wp.merge(pr, on=["race_id", "horse_id"], how="inner")
m["pb"] = pd.cut(m["p_win"], [0, 0.01, 0.02, 0.035, 0.05, 0.08, 0.15, 1], labels=["<1%", "1-2%", "2-3.5%", "3.5-5%", "5-8%", "8-15%", "15%+"])
BYP = m.groupby(["pb", "year"], observed=True)["win_odds"].median().unstack()
display(BYP.round(1)); BYP.to_csv(OUT / "median_odds_by_model_p.csv")
print("2022 → 2025 の変化率:", ((BYP[2025] / BYP[2022] - 1) * 100).round(1).to_dict())

## 最終オッズ予測のローリング評価
検証年 Y の予測 = Y 年より前（Y-01-01 の 10 日前まで）で学習。目的変数 = log(単勝の確定オッズ)。

In [ ]:
D = tp05_data.load_dataset(WORK)
df = D["df"]
feats = D["FEATS"]["base+S+C1"]
T = df[["race_id", "horse_id", "date"]].merge(wp[["race_id", "horse_id", "win_odds"]], on=["race_id", "horse_id"], how="inner")
idx = df.reset_index().merge(T[["race_id", "horse_id", "win_odds"]], on=["race_id", "horse_id"], how="inner")
X = idx[feats]
y = np.log(idx["win_odds"].to_numpy())
dates = pd.to_datetime(idx["date"])
yr = dates.dt.year.to_numpy()
race = idx["race_id"].to_numpy()
print(f"行 {len(idx):,}  特徴量 {len(feats)}")
P = {"objective": "regression", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 50, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
     "verbosity": -1, "num_threads": THREADS, "seed": 0, "deterministic": True, "force_row_wise": True}
VARIANTS = {"一様": dict(), "半減期1年": dict(hl=1.0), "半減期2年": dict(hl=2.0), "直近2年": dict(window=2), "直近3年": dict(window=3)}
PRED = {}
rows = []
for Y in (2022, 2023, 2024, 2025):
    tr = np.flatnonzero(dates < pd.Timestamp(f"{Y}-01-01") - pd.Timedelta(days=10))
    te = np.flatnonzero(yr == Y)
    for name, v in VARIANTS.items():
        t0 = time.time()
        sel = tr[yr[tr] >= Y - v["window"]] if "window" in v else tr
        w = 0.5 ** (((Y - 1) - yr[sel]) / v["hl"]) if "hl" in v else None
        b = lgb.train(P, lgb.Dataset(X.iloc[sel], label=y[sel], weight=w), num_boost_round=500)
        p = b.predict(X.iloc[te])
        PRED[(Y, name)] = (te, p)
        r = y[te] - p
        rows.append({"検証年": Y, "重み": name, "rmse_log": float(np.sqrt((r ** 2).mean())), "mae_log": float(np.abs(r).mean()), "偏り(実測−予測)": float(r.mean()),
                     "偏り(予測>=30倍)": float(r[np.exp(p) >= 30].mean()), "偏り(予測<5倍)": float(r[np.exp(p) < 5].mean()), "sec": time.time() - t0})
        gc.collect()
    print(Y, "済", flush=True)
R = pd.DataFrame(rows)
display(R.pivot(index="重み", columns="検証年", values="rmse_log").round(4))
display(R.pivot(index="重み", columns="検証年", values="偏り(実測−予測)").round(4))
display(R.pivot(index="重み", columns="検証年", values="偏り(予測>=30倍)").round(4))
R.to_csv(OUT / "rolling_eval.csv", index=False)

In [ ]:
# 一様な重みとの差（二乗誤差。レース単位の paired bootstrap）。負なら重み付きの方が良い
rng = np.random.default_rng(0)
rows = []
for Y in (2022, 2023, 2024, 2025):
    te, p0 = PRED[(Y, "一様")]
    e0 = (y[te] - p0) ** 2
    codes = pd.factorize(race[te])[0]
    for name in VARIANTS:
        if name == "一様":
            continue
        _, p1 = PRED[(Y, name)]
        d = (y[te] - p1) ** 2 - e0
        per = pd.Series(d).groupby(codes).mean().to_numpy()
        bs = np.array([per[rng.integers(0, len(per), len(per))].mean() for _ in range(500)])
        rows.append({"検証年": Y, "重み": name, "二乗誤差の差": float(per.mean()), "95%下限": float(np.quantile(bs, 0.025)), "95%上限": float(np.quantile(bs, 0.975))})
DIFF = pd.DataFrame(rows)
DIFF["改善"] = DIFF["95%上限"] < 0
display(DIFF.pivot(index="重み", columns="検証年", values="二乗誤差の差").round(5))
display(DIFF.pivot(index="重み", columns="検証年", values="改善"))
DIFF.to_csv(OUT / "weights_vs_uniform.csv", index=False)
good = DIFF.groupby("重み")["改善"].sum()
min_eff = DIFF.groupby("重み")["二乗誤差の差"].mean()
print("改善した検証年の数:", good.to_dict()); print("平均の二乗誤差の差:", min_eff.round(5).to_dict())
BEST = None
for name in good.sort_values(ascending=False).index:
    if good[name] >= 3 and min_eff[name] < -0.001:
        BEST = name; break
print("提案する重み:", BEST)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "market_structure": STRUCT.round(4).reset_index().to_dict("records"), "median_odds_by_model_p": BYP.round(2).reset_index().astype(str).to_dict("records"),
                                             "rolling": R.round(5).to_dict("records"), "weights_vs_uniform": DIFF.round(6).to_dict("records"), "proposal": BEST}, ensure_ascii=False, indent=1, default=str))
with mlflow.start_run(run_name="tp-10-odds-drift", tags={"notebook": "tp-10", "git_commit": GIT_COMMIT}):
    for _, r in R.iterrows():
        mlflow.log_metrics({f"rmse_log__{r['重み']}__{int(r['検証年'])}": float(r["rmse_log"])})
    for f_ in ("market_structure_by_year.csv", "calibration_by_band_year.csv", "median_odds_by_model_p.csv", "rolling_eval.csv", "weights_vs_uniform.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "odds_drift")